In [ ]:
import os 
import base64
from openai import AzureOpenAI
import json
import re
import numpy as np
import datetime
import pandas as pd
import time
import ast
from collections import defaultdict

from pypdf import PdfReader
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_openai import AzureOpenAIEmbeddings


import requests
from azure.identity import DefaultAzureCredential, get_bearer_token_provider


In [ ]:
endpoint = "<your_azure_endpoint_here>"
model_name = "gpt-4o"
deployment = "gpt-4o"

subscription_key = "<your_subscription_key_here>"
api_version = "<your_api_version_here>"

client = AzureOpenAI(
    api_version=api_version,
    azure_endpoint=endpoint,
    api_key=subscription_key,
)

In [ ]:
df_note = pd.read_csv("note_data.csv", encoding='utf-8')
df_note.reset_index(drop=True, inplace=True)

In [ ]:
with open("gira_reference.txt", "r", encoding="utf-8") as f:
    GIRA_guideline = f.read()

**1. Guideline-aware prompt - direct input**

In [ ]:
def build_gira_prompt(note_text: str, GIRA_guideline: str) -> str:  
    """  
    Builds the user prompt for extracting GIRA-informed clinical actions.  
    """  
    return f"""  
TASK:
Using ONLY the MEDICAL NOTE {note_text}, identify clinical actions that are explicitly documented as being initiated, ordered, or performed **in response to genetic findings or a Genome-informed Risk Assessment (GIRA)**.
Use the GIRA-RELATED RECOMMENDATION CONTEXT {GIRA_guideline} ONLY to determine whether an action qualifies as GIRA-informed and to classify the action type.

DEFINITION:
A GIRA-informed clinical action is one that the MEDICAL NOTE explicitly links to:
–	a genetic result, variant, mutation, polygenic risk, calculated risk (e.g., BOADICEA), 
–	a documented Genome-Informed Risk Assessment or interpretation.

Rules
–	The MEDICAL NOTE is the ONLY evidence that an action occurred.
–	The MEDICAL NOTE must explicitly indicate a genetic or GIRA-related reason for the action.
–	Do NOT infer causality, assume intent, or recommend actions.
–	Only consider genetic results related to the following conditions:
▪	Asthma
▪	Atrial fibrillation
▪	Breast cancer
▪	Chronic kidney disease
▪	Coronary heart disease
▪	Hypercholesterolemia
▪	Obesity / BMI
▪	Prostate cancer
▪	Type 1 diabetes
▪	Type 2 diabetes

Action Types (GIRA-Aligned)
–	referral
–	lab_test
–	imaging_or_monitoring

Genetic-Related Specialist Referrals
Only classify an action as a referral if the MEDICAL NOTE explicitly documents referral to one of the following AND links it to a genetic or GIRA-related reason:

–	Oncologist
–	Surgeon (oncology-related)
–	Cardiologist
–	Electrophysiologist (cardiology)
–	Lipids Specialist (cardiology)
–	Nephrologist
–	Endocrinologist
–	Gynecologist
–	Urologist
–	Dietician
–	Nutritionist
–	Genetic Counselor
Extraction
For each GIRA-informed action explicitly documented, extract:

–	action_type
–	action_name (e.g., cardiology referral, lipid panel, ECG, renal ultrasound)
–	evidence_citation (verbatim text demonstrating both the action and its genetic/GIRA-informed reason)

Output (JSON Only)

{
  "actions_present": true or false,
  "actions": [
    {
      "action_type": string,
      "action_name": string,
      "evidence_citation": string
    }
    ...
    /* repeat for additional clinical actions if applicable */
  ]
}


RULES:
- The MEDICAL NOTE is the ONLY evidence that an action occurred.
- The MEDICAL NOTE must explicitly indicate a genetic or GIRA-related reason for the action.
- Use the GIRA guideline ONLY to classify the action type, NOT to infer missing actions.
- Do NOT infer causality, assume intent, or recommend actions.
- Only consider genetic results related to the following conditions:
    - Asthma
    - Atrial fibrillation
    - Breast cancer
    - Chronic kidney disease
    - Coronary heart disease
    - Hypercholesterolemia
    - Obesity / BMI
    - Prostate cancer
    - Type 1 diabetes
    - Type 2 diabetes

ACTION TYPES (GIRA-ALIGNED):
- referral
- lab_test
- imaging_or_monitoring

GENETIC-RELATED SPECIALIST REFERRALS:
Only classify an action as a referral if the MEDICAL NOTE explicitly documents referral to one of the following AND links it to a genetic or GIRA-related reason:
- Oncologist
- Surgeon (oncology-related)
- Cardiologist
- Electrophysiologist (cardiology)
- Lipids Specialist (cardiology)
- Nephrologist
- Endocrinologist
- Gynecologist
- Urologist
- Dietician
- Nutritionist
- Genetic Counselor

EXTRACTION:
For each GIRA-informed action explicitly documented, extract:
- action_type
- action_name (e.g., cardiology referral, lipid panel, ECG, renal ultrasound)
- evidence_citation (verbatim text demonstrating both the action and its genetic/GIRA-informed)

OUTPUT (JSON ONLY):
{{
  "actions_present": true or false,
  "actions": [
    {{
      "action_type": string,
      "action_name": string,
      "evidence_citation": string
    }}
        ...
        /* repeat for additional clinical actions if applicable */
  ]
}}
"""  

In [ ]:
def run_GIRA_extraction(  
    df_note,  
    client,  
    deployment,  
    GIRA_guideline,  
    output_path="GPT4o_GIRA_direct_input.json"  
):  
    start_time = time.perf_counter()

    system_message = (  
        "You are a medical expert. Your task is to analyze a medical note and "  
        "extract genetic-informed clinical actions in a structured JSON format. "  
        "Do not include any explanations, disclaimers, or text outside of the JSON object."  
    )  
  
    all_outputs = []  
  
    for i, row in enumerate(df_note.itertuples(index=False)):  
  
        prompt = build_gira_prompt(  
            note_text=row.note_text,  
            GIRA_guideline=GIRA_guideline  
        )  
  
        try:  
            completion = client.chat.completions.create(  
                model=deployment,  
                messages=[  
                    {"role": "system", "content": system_message},  
                    {"role": "user", "content": prompt},  
                ],  
                temperature=0,  
                max_tokens=4096,  
            )  
  
            content = completion.choices[0].message.content.strip()  
  

            if content.startswith("```") and content.endswith("```"):  
                content = content[3:-3].strip()  
            if content.lower().startswith("json"):  
                content = content[4:].strip()  
  
            try:  
                parsed = json.loads(content)  
  
                # ---- Attach metadata ----  
                parsed["person"] = int(row.MRN)  
                parsed["note_date"] = str(row.note_date)  
                parsed["note_id"] = int(row.note_id)  
                parsed["note_title"] = str(row.note_title)  
                parsed["GIRA_date"] = str(row.GIRA_date)  
  
                all_outputs.append(parsed)  
  
            except json.JSONDecodeError:  
                print(f"⚠️ JSON parse error at index {i}")  
                all_outputs.append({  
                    "error": "Invalid JSON",  
                    "index": i,  
                    "raw": content  
                })  
  
        except Exception as e:  
            print(f"❌ Model error at index {i}: {e}")  
            all_outputs.append({  
                "error": "Model error",  
                "index": i,  
                "exception": str(e)  
            })  
  
  
    with open(output_path, "w", encoding="utf-8") as f:  
        json.dump(all_outputs, f, ensure_ascii=False, indent=2)  

    elapsed = time.perf_counter() - start_time  
    hours = elapsed / 3600
  
    print(f"✅ GIRA-informed clinical action extraction completed. {len(all_outputs)} records written to {output_path}")  
    print(f"✅ Completed in {hours:.3f} hours")  

In [ ]:
run_GIRA_extraction(  
    df_note=df_note,  
    client=client,  
    deployment=deployment,  
    GIRA_guideline=GIRA_guideline  
)  

****2. Guideline-aware prompt - RAG****

In [ ]:
client = AzureOpenAI(
azure_endpoint = endpoint,
api_key= subscription_key,
api_version="<API_VERSION>"
)

assistant = client.beta.assistants.create(
model="gpt-4o", 
instructions="You are a medical expert. Your task is to analyze a medical note and extract genetic-informed clinical actions based on the attached Genetic-informed Risk Assessment (GIRA) guidelines in a structured JSON format. Do not include any explanations, disclaimers, or text outside of the JSON object.",
tools=[{"type":"file_search"}],
tool_resources={"file_search":{"vector_store_ids":["xxxxxxxxxx"]}},
temperature=0,
top_p=1
)

In [ ]:
def extract_text_from_message(message):  
    texts = []  
    for block in message.content:  
        if block.type == "text" and hasattr(block, "text"):  
            texts.append(block.text.value)  
    return texts  

In [ ]:

MAX_POLL_SECONDS = 60
POLL_INTERVAL = 1.0
OUTFILE = "GPT4o_GIRA_RAG.json"
# ------------------------------------

start_time = time.perf_counter()

def get_messages_list(messages_obj):
    """Return a python list of message objects from many possible SDK shapes."""
  
    if hasattr(messages_obj, "data"):
        return getattr(messages_obj, "data") or []
  
    if isinstance(messages_obj, dict) and "data" in messages_obj:
        return messages_obj["data"] or []
  
    if isinstance(messages_obj, list):
        return messages_obj
  
    return [messages_obj] if messages_obj else []

# ---------- Main loop ----------
all_outputs = []

for i, row in enumerate(df_note.itertuples(index=False)):

    note_text = getattr(row, "note_text", "")
    prompt = f"""TASK:
Using ONLY the MEDICAL NOTE {note_text}, identify clinical actions that are explicitly documented as being initiated, ordered, or performed **in response to genetic findings or a Genome-informed Risk Assessment (GIRA)**.
Use the GIRA-RELATED RECOMMENDATION CONTEXT from the provided pdf ONLY to determine whether an action qualifies as GIRA-informed and to classify the action type.

DEFINITION:
A GIRA-informed clinical action is one that the MEDICAL NOTE explicitly links to:
- a genetic result, variant, mutation, polygenic risk, or
- a documented genetic-informed risk assessment or interpretation.

RULES:
- The MEDICAL NOTE is the ONLY evidence that an action occurred.
- The MEDICAL NOTE must explicitly indicate a genetic or GIRA-related reason for the action.
- Use the GIRA guideline ONLY to classify the action type, NOT to infer missing actions.
- Do NOT infer causality, assume intent, or recommend actions.
- Only consider genetic results related to the following conditions:
    - Asthma
    - Atrial fibrillation
    - Breast cancer
    - Chronic kidney disease
    - Coronary heart disease
    - Hypercholesterolemia
    - Obesity / BMI
    - Prostate cancer
    - Type 1 diabetes
    - Type 2 diabetes

ACTION TYPES (GIRA-ALIGNED):
- referral
- lab_test
- imaging_or_monitoring

GENETIC-RELATED SPECIALIST REFERRALS:
Only classify an action as a referral if the MEDICAL NOTE explicitly documents referral to one of the following AND links it to a genetic or GIRA-related reason:
- Oncologist
- Surgeon (oncology-related)
- Cardiologist
- Electrophysiologist (cardiology)
- Lipids Specialist (cardiology)
- Nephrologist
- Endocrinologist
- Gynecologist
- Urologist
- Dietician
- Nutritionist
- Genetic Counselor

EXTRACTION:
For each GIRA-informed action explicitly documented, extract:
- action_type
- action_name (e.g., cardiology referral, lipid panel, ECG, renal ultrasound)
- evidence_citation (verbatim text demonstrating both the action and its genetic/GIRA-informed)

OUTPUT (JSON ONLY):
{{
  "actions_present": true or false,
  "actions": [
    {{
      "action_type": string,
      "action_name": string,
      "evidence_citation": string
    }}
        ...
        /* repeat for additional clinical actions if applicable */
  ]
}}

""" 

    try:

        thread = client.beta.threads.create()

        client.beta.threads.messages.create(
            thread_id=thread.id,
            role="user",
            content=prompt
        )

        run = client.beta.threads.runs.create(thread_id=thread.id, assistant_id=assistant.id)

        start_t = time.time()
        while run.status in ("queued", "in_progress", "cancelling"):
            if time.time() - start_t > MAX_POLL_SECONDS:
                raise TimeoutError(f"Thread run timed out after {MAX_POLL_SECONDS}s (status={run.status})")
            time.sleep(POLL_INTERVAL)
            run = client.beta.threads.runs.retrieve(thread_id=thread.id, run_id=run.id)

        if run.status == "completed":
            messages = client.beta.threads.messages.list(thread_id=thread.id)
            message_list = get_messages_list(messages)

            assistant_texts = []
            for m in message_list:
                role = getattr(m, "role", None) if not isinstance(m, dict) else m.get("role")
                assistant_id = getattr(m, "assistant_id", None) if not isinstance(m, dict) else m.get("assistant_id")
                if role == "assistant" or assistant_id:
                    t = extract_text_from_message(m)
                    if t:
                        assistant_texts.append(t)

            if not assistant_texts:
                all_outputs.append({"error": "No assistant message found", "index": i, "run_status": run.status})
                continue

            content = assistant_texts[0][0]

            if content.startswith("```") and content.endswith("```"):
                content = content[3:-3].strip()
            if content.lower().startswith("json"):
                content = content[len("json"):].strip()

            try:
                parsed = json.loads(content)
                # Attach your metadata
                parsed["person"] = int(getattr(row, "MRN", 0))
                parsed["note_date"] = str(getattr(row, "note_date", ""))
                parsed["note_id"] = int(getattr(row, "note_id", 0))
                parsed["note_title"] = str(getattr(row, "note_title", ""))
                parsed["GIRA_date"] = str(getattr(row, "GIRA_date", ""))
                all_outputs.append(parsed)
            except json.JSONDecodeError:
                print(f"Warning: JSON decode error at row {i}. Raw assistant output:\n{content}")
                all_outputs.append({"error": "Invalid JSON", "index": i, "raw": content})

        elif run.status == "requires_action":

            print(f"Run requires action for row {i}.")
            all_outputs.append({"error": "requires_action", "index": i, "run": run})
        else:
            print(f"Run finished with unexpected status '{run.status}' at row {i}.")
            all_outputs.append({"error": f"run_status_{run.status}", "index": i, "run": run})

    except Exception as e:
        print(f"Error during inference at index {i}: {e}")
        all_outputs.append({"error": "Model error", "index": i, "exception": str(e)})


with open(OUTFILE, "w", encoding="utf-8") as f:
    json.dump(all_outputs, f, ensure_ascii=False, indent=2)


elapsed = time.perf_counter() - start_time  
hours = elapsed / 3600

print(f"✅ GIRA-informed clinical action extraction completed. {len(all_outputs)} records written to {OUTFILE}")  
print(f"✅ Completed in {hours:.3f} hours")  



****3. Load output JSON file****

In [ ]:
def safe_json_loads(raw_str):
    try:
        return json.loads(raw_str)
    except json.JSONDecodeError:
        try:
            fixed = raw_str.replace("\\\'", "'") 
            return json.loads(fixed)
        except Exception as e2:
            return None

with open('GPT4o_GIRA_RAG.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

rows = []
for entry in data:
    parsed = None

    if 'error' in entry and 'raw' in entry:
        parsed = safe_json_loads(entry['raw'])
        index = entry.get("index")
        if parsed and parsed.get("actions_present") == True:
            entry = parsed
            person = int(df_note.loc[index, "MRN"])
            note_date = str(df_note.loc[index, "note_date"])
            GIRA_date = str(df_note.loc[index, "GIRA_date"])
            note_title = str(df_note.loc[index, "note_title"])
            note_id = int(df_note.loc[index, "note_id"])
            
        else:
            print(f"Failed to parse index: {index}")

            continue
            
    elif entry.get("actions_present") == True:
        parsed = entry
        person = parsed.get("person")
        note_date = parsed.get("note_date")
        GIRA_date = parsed.get("GIRA_date")
        note_title = parsed.get("note_title")
        note_id = parsed.get("note_id")
    else:
        continue

    actions = parsed.get("actions", [])
    for action in actions:
        row = {
            "person": person,
            "GIRA_date": GIRA_date,
            "note_date": note_date,
            "note_id": note_id,
            "note_title": note_title,
            "action_type": action.get("action_type"),
            "action_name": action.get("action_name"),
            "evidence_citation": action.get("evidence_citation")
            
        }
        rows.append(row)

      
df = pd.DataFrame(rows)